Activity 4: aggregating everything

Actual training

In [ ]:
# Actual training - Classification Model
import os
import random
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# -------------------------------------------------------------
# 1. Pure NumPy Macro F1 Score (No sklearn needed!)
# -------------------------------------------------------------
def compute_macro_f1(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-7) -> float:
    tp = np.sum((y_true == 1) & (y_pred == 1), axis=0)
    fp = np.sum((y_true == 0) & (y_pred == 1), axis=0)
    fn = np.sum((y_true == 1) & (y_pred == 0), axis=0)
    
    f1_per_class = (2 * tp) / (2 * tp + fp + fn + eps)
    return float(np.mean(f1_per_class))

# -------------------------------------------------------------
# 2. Device & Seed Configuration
# -------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cpu':
    num_threads = min(16, os.cpu_count() or 4)
    torch.set_num_threads(num_threads)
    print(f"PyTorch CPU threads: {num_threads}")

# Paths
ROOT = Path(".").resolve()
from data_utils import ensure_dataset
DATA_DIR = ensure_dataset(ROOT / "data")  # downloads from Google Drive if missing
TRAIN_IMG_DIR = DATA_DIR / "train" / "images"
LABELS_CSV = DATA_DIR / "train" / "labels.csv"

WEIGHTS_DIR = Path("classification/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------
# 3. Classes & Dataset Definition
# -------------------------------------------------------------
CLASSES: List[str] = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

class MultiLabelDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.labels = self.df[CLASSES].values.astype(np.float32)
        self.image_ids = self.df['image_id'].values

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_path = self.img_dir / f"{self.image_ids[idx]}.jpg"
        with Image.open(img_path) as img:
            image = img.convert("RGB")
            
        if self.transform:
            image = self.transform(image)
            
        target = torch.from_numpy(self.labels[idx])
        return image, target

# -------------------------------------------------------------
# 4. Transforms & DataLoaders
# -------------------------------------------------------------
IMAGE_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

df_all = pd.read_csv(LABELS_CSV)
val_sample = df_all.sample(frac=0.2, random_state=SEED)
train_sample = df_all.drop(val_sample.index)

train_dataset = MultiLabelDataset(train_sample, TRAIN_IMG_DIR, transform=train_transforms)
val_dataset = MultiLabelDataset(val_sample, TRAIN_IMG_DIR, transform=val_transforms)

BATCH_SIZE = 32 if device.type == 'cuda' else 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Dataset Split: {len(train_dataset)} Train | {len(val_dataset)} Validation")

# -------------------------------------------------------------
# 5. Model Architecture (ResNet-50 + Custom Head)
# -------------------------------------------------------------
class ClassifierNet(nn.Module):
    def __init__(self, num_classes: int = 20):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if hasattr(models, 'ResNet50_Weights') else None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)
        feat = self.pool(feat).flatten(1)
        return self.head(feat)

model = ClassifierNet(num_classes=len(CLASSES)).to(device)

# -------------------------------------------------------------
# 6. Loss, Optimizer & Scheduler
# -------------------------------------------------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
EPOCHS = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# -------------------------------------------------------------
# 7. Training Loop
# -------------------------------------------------------------
best_val_f1 = 0.0
best_model_path = WEIGHTS_DIR / "best_classifier.pth"

print(f"\nStarting training for {EPOCHS} epochs...\n" + "-"*50)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    train_loss /= len(train_dataset)
    scheduler.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    all_targets, all_preds = [], []
    
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).int()
            
            all_targets.append(targets.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            
    val_loss /= len(val_dataset)
    val_f1 = compute_macro_f1(np.vstack(all_targets), np.vstack(all_preds))
    
    is_best = val_f1 > best_val_f1
    if is_best:
        best_val_f1 = val_f1
        torch.save({
            'state_dict': model.state_dict(),
            'val_f1': val_f1,
            'epoch': epoch
        }, best_model_path)
    
    best_mark = " [BEST SAVED]" if is_best else ""
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Mean F1: {val_f1:.4f}{best_mark}")

print("-" * 50)
print(f"Training Complete! Best Validation Mean F1: {best_val_f1:.4f}")
print(f"Saved weights to: {best_model_path}")

In [ ]:
# model testing: classification
import numpy as np
from PIL import Image
from pathlib import Path
from classification.model import ClassificationModel

# 1. Instantiate using your newly trained weights
clf = ClassificationModel(weights_dir="classification/weights")

# 2. Pick a test image
sample_path = Path("data/Dataset/train/images/img_00003.jpg")
with Image.open(sample_path) as img:
    img_arr = np.array(img.convert("RGB"))

# 3. Predict class probabilities
probs_dict = clf.predict(img_arr)

# 4. Filter classes with probability >= 0.5 (per assignment rules)
predicted_classes = [cls for cls, prob in probs_dict.items() if prob >= 0.5]

print(f"Image: {sample_path.name}")
print(f"Top predicted probabilities:")
sorted_probs = sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)[:5]
for cls, prob in sorted_probs:
    print(f"  - {cls:12s}: {prob:.4f}")

print(f"\nFinal Classification Output: '{' '.join(predicted_classes) if predicted_classes else 'background'}'")

# Segmentation

#### Activity 1: Ground Truth Masks & The "255" Ignore Label

In semantic segmentation benchmarks (like PASCAL VOC), images have object boundaries marked with a special value 255 (void / boundary).

If we treated 255 as a normal class, the model would fail or crash with an index error since our output layer only has 21 channels (
0
0 through 
20
20).
PyTorch provides nn.CrossEntropyLoss(ignore_index=255), which masks out those pixels during gradient calculation.

Actual Segmentation

In [ ]:
# Actual training - segmentation model

import os
import random
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# -------------------------------------------------------------
# 1. Device & Reproducibility
# -------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cpu':
    num_threads = min(16, os.cpu_count() or 4)
    torch.set_num_threads(num_threads)
    print(f"Configured PyTorch CPU threads: {num_threads}")

# Paths
ROOT = Path(".").resolve()
from data_utils import ensure_dataset
DATA_DIR = ensure_dataset(ROOT / "data")  # downloads from Google Drive if missing
TRAIN_IMG_DIR = DATA_DIR / "train" / "images"
TRAIN_MASK_DIR = DATA_DIR / "train" / "segmentation_masks"

WEIGHTS_DIR = Path("segmentation/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------
# 2. Segmentation Dataset with Nearest-Neighbor Masks
# -------------------------------------------------------------
NUM_CLASSES = 21  # 20 objects + 1 background (0)
IMAGE_SIZE = 224   # Efficient spatial resolution for fast convergence

class SegmentationDataset(Dataset):
    def __init__(self, image_paths: List[Path], mask_dir: Path, img_size: int = IMAGE_SIZE, is_train: bool = True):
        self.image_paths = image_paths
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.is_train = is_train
        
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_p = self.image_paths[idx]
        mask_p = self.mask_dir / f"{img_p.stem}.png"
        
        # Load image & mask
        img = Image.open(img_p).convert("RGB")
        mask = Image.open(mask_p)
        
        # Resize: Bilinear for image, NEAREST for discrete class mask!
        img = img.resize((self.img_size, self.img_size), Image.Resampling.BILINEAR)
        mask = mask.resize((self.img_size, self.img_size), Image.Resampling.NEAREST)
        
        # Random horizontal flip during training
        if self.is_train and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            
        # Convert to tensors
        img_tensor = transforms.ToTensor()(img)
        img_tensor = self.normalize(img_tensor)
        
        mask_tensor = torch.from_numpy(np.array(mask, dtype=np.int64))
        return img_tensor, mask_tensor

# Train/Val Split (80% Train, 20% Val)
all_images = sorted(TRAIN_IMG_DIR.glob("*.jpg"))
random.Random(SEED).shuffle(all_images)

split_idx = int(0.8 * len(all_images))
train_imgs = all_images[:split_idx]
val_imgs = all_images[split_idx:]

train_dataset = SegmentationDataset(train_imgs, TRAIN_MASK_DIR, is_train=True)
val_dataset = SegmentationDataset(val_imgs, TRAIN_MASK_DIR, is_train=False)

BATCH_SIZE = 16 if device.type == 'cuda' else 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Segmentation Split: {len(train_dataset)} Train | {len(val_dataset)} Validation")

# -------------------------------------------------------------
# 3. Model Architecture: ResNet-50 + Multi-Scale Skip Decoder
# -------------------------------------------------------------
class ConvBNAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size // 2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class CustomSegmentationNet(nn.Module):
    def __init__(self, num_classes: int = 21):
        super().__init__()
        # Backbone (ResNet-50)
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if hasattr(models, 'ResNet50_Weights') else None)
        self.stem = nn.Sequential(
            backbone.conv1, 
            backbone.bn1, 
            backbone.relu, 
            backbone.maxpool
        )
        self.layer1 = backbone.layer1  # 256 channels (c1)
        self.layer2 = backbone.layer2  # 512 channels (c2)
        self.layer3 = backbone.layer3  # 1024 channels (c3)
        self.layer4 = backbone.layer4  # 2048 channels (c4)
        
        # Decoder Fusing Stages
        self.proj4 = ConvBNAct(2048, 256, kernel_size=1)
        self.fuse34 = nn.Sequential(ConvBNAct(256 + 1024, 256), ConvBNAct(256, 256))
        self.fuse23 = nn.Sequential(ConvBNAct(256 + 512, 192), ConvBNAct(192, 192))
        self.fuse12 = nn.Sequential(ConvBNAct(192 + 256, 128), ConvBNAct(128, 128))
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = x.shape[2], x.shape[3]
        # Encoder
        x_stem = self.stem(x)
        c1 = self.layer1(x_stem)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        
        # Decoder with Skip Connections
        d = self.proj4(c4)
        d = F.interpolate(d, size=c3.shape[2:], mode='bilinear', align_corners=False)
        d = self.fuse34(torch.cat([d, c3], dim=1))
        
        d = F.interpolate(d, size=c2.shape[2:], mode='bilinear', align_corners=False)
        d = self.fuse23(torch.cat([d, c2], dim=1))
        
        d = F.interpolate(d, size=c1.shape[2:], mode='bilinear', align_corners=False)
        d = self.fuse12(torch.cat([d, c1], dim=1))
        
        # Final logits resized back to input resolution
        logits = self.classifier(d)
        logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        return logits

model = CustomSegmentationNet(num_classes=NUM_CLASSES).to(device)

# -------------------------------------------------------------
# 4. Composite Loss: Cross-Entropy + Soft Dice Loss
# -------------------------------------------------------------
class DiceLoss(nn.Module):
    def __init__(self, num_classes: int = 21, ignore_index: int = 255, smooth: float = 1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        valid_mask = (targets != self.ignore_index)
        targets_clean = torch.where(valid_mask, targets, torch.zeros_like(targets))
        targets_one_hot = F.one_hot(targets_clean, num_classes=self.num_classes).permute(0, 3, 1, 2).float()
        
        valid_mask_exp = valid_mask.unsqueeze(1).float()
        probs = probs * valid_mask_exp
        targets_one_hot = targets_one_hot * valid_mask_exp
        
        intersection = torch.sum(probs * targets_one_hot, dim=(0, 2, 3))
        cardinality = torch.sum(probs + targets_one_hot, dim=(0, 2, 3))
        
        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - torch.mean(dice_score)

criterion_ce = nn.CrossEntropyLoss(ignore_index=255)
criterion_dice = DiceLoss(num_classes=NUM_CLASSES, ignore_index=255)

def composite_loss(logits, targets):
    return 0.5 * criterion_ce(logits, targets) + 0.5 * criterion_dice(logits, targets)

# -------------------------------------------------------------
# 5. Optimizer, Scheduler & Evaluation Metric (mIoU)
# -------------------------------------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
EPOCHS = 12
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

def compute_batch_miou(preds: np.ndarray, targets: np.ndarray, ignore_index: int = 255) -> float:
    batch_ious = []
    for p, t in zip(preds, targets):
        valid = (t != ignore_index)
        p_val, t_val = p[valid], t[valid]
        present = np.unique(np.concatenate([p_val, t_val]))
        img_ious = []
        for c in present:
            inter = np.sum((p_val == c) & (t_val == c))
            union = np.sum((p_val == c) | (t_val == c))
            if union > 0:
                img_ious.append(inter / union)
        if img_ious:
            batch_ious.append(np.mean(img_ious))
    return float(np.mean(batch_ious)) if batch_ious else 1.0

# -------------------------------------------------------------
# 6. Training & Validation Loop
# -------------------------------------------------------------
best_val_miou = 0.0
best_model_path = WEIGHTS_DIR / "best_segmentation.pth"

print(f"\nStarting Segmentation training for {EPOCHS} epochs...\n" + "-"*55)

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        loss = composite_loss(logits, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    train_loss /= len(train_dataset)
    scheduler.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_miou_list = []
    
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            logits = model(images)
            loss = composite_loss(logits, masks)
            val_loss += loss.item() * images.size(0)
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            targets_np = masks.cpu().numpy()
            val_miou_list.append(compute_batch_miou(preds, targets_np))
            
    val_loss /= len(val_dataset)
    val_miou = float(np.mean(val_miou_list))
    
    is_best = val_miou > best_val_miou
    if is_best:
        best_val_miou = val_miou
        torch.save({
            'state_dict': model.state_dict(),
            'val_miou': val_miou,
            'epoch': epoch
        }, best_model_path)
        
    best_mark = " [BEST SAVED]" if is_best else ""
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val mIoU: {val_miou*100:.2f}%{best_mark}")

print("-" * 55)
print(f"Training Complete! Best Validation mIoU: {best_val_miou*100:.2f}%")
print(f"Saved weights to: {best_model_path}")